# 02 · Parallelism and resilience

The patterns in this notebook cover most day-to-day pipeline architecture questions:
fan-out/fan-in, bounding parallelism, retries and timeouts, recovering from OOMs,
checkpointing long tasks, and pausing for human approval.

**Learning goals**

1. Fan out work with `asyncio.gather` and `flyte.map.aio` — and know when to use which
2. Bound parallelism with semaphores and `concurrency`
3. Configure retries and two-dimensional timeouts (runtime vs queue wait)
4. Recover from OOM by retrying with more memory — *in code*
5. Checkpoint inside a task with `@flyte.trace`
6. Gate a pipeline on an external signal (human approval) with `flyte.new_condition`

In [ ]:
import asyncio
from typing import List

import flyte

flyte.init_from_config()

env = flyte.TaskEnvironment(
    name="parallel_patterns",
    resources=flyte.Resources(cpu="1", memory="1Gi"),
)

## 1. Fan-out / fan-in

In v2 there is no `@dynamic` and no special map-task construct to learn: fan-out is plain
Python. Two idioms:

- **`asyncio.gather`** — explicit and flexible; you build the list of coroutines yourself.
  Best for heterogeneous fan-out (different tasks, conditional branches).
- **`flyte.map.aio`** — for homogeneous "same task over a list" fan-out; streams results as
  they finish, groups the actions in the UI, and has first-class partial-failure handling.

In [ ]:
@env.task
async def score(item: str) -> float:
    # stand-in for a real per-item computation
    return float(len(item))


@env.task
async def fanout_gather(items: List[str]) -> float:
    scores = await asyncio.gather(*[score(item=i) for i in items])
    return sum(scores) / len(scores)


run = await flyte.run.aio(fanout_gather, items=["alpha", "beta", "gamma", "delta"])
print(run.url)
await run.wait.aio()
run.outputs()

In [ ]:
@env.task
async def fanout_map(items: List[str]) -> List[float]:
    results: List[float] = []
    # return_exceptions=True: one failed item doesn't cancel the other in-flight items.
    # Without it, the first failure aborts the whole map.
    async for r in flyte.map.aio(score, items, return_exceptions=True):
        if isinstance(r, Exception):
            print(f"item failed: {r!r}")
            continue                      # or collect for a retry pass
        results.append(r)
    return results

### Binding constants with `functools.partial`

`flyte.map` iterates over **one** argument. Bind the constant ones first:

In [ ]:
from functools import partial


@env.task
async def score_with_model(item: str, model_version: str, threshold: float) -> bool:
    return float(len(item)) * (1.0 if model_version == "v2" else 0.5) > threshold


@env.task
async def batch(items: List[str]) -> List[bool]:
    fn = partial(score_with_model, model_version="v2", threshold=3.0)
    out: List[bool] = []
    async for r in flyte.map.aio(fn, items, return_exceptions=True):
        if isinstance(r, Exception):
            raise r
        out.append(r)
    return out

### Grouping actions in the UI

Big fan-outs make the run view noisy. `flyte.group` (or `group_name=` on `flyte.map`)
collapses related actions into a labelled group:

In [ ]:
@env.task
async def staged_pipeline(items: List[str]) -> List[float]:
    with flyte.group("preprocessing"):
        cleaned = await asyncio.gather(*[score(item=i) for i in items])

    with flyte.group("scoring"):
        rescored = await asyncio.gather(*[score(item=str(c)) for c in cleaned])

    return list(rescored)

## 2. Bounding parallelism

Unbounded `gather` over 100k coroutines will overwhelm your own task (memory) and whatever
you're calling (rate limits) before the cluster is the bottleneck. Two levers:

- `flyte.map.aio(fn, items, concurrency=N)` — at most N child actions in flight
- `asyncio.Semaphore` — same idea for hand-rolled `gather`, also works around *non-task*
  calls (HTTP APIs) inside a single task

In [ ]:
@env.task
async def rate_limited(items: List[str], max_concurrent: int = 10) -> List[float]:
    sem = asyncio.Semaphore(max_concurrent)

    async def bounded(i: str) -> float:
        async with sem:
            return await score(item=i)

    return list(await asyncio.gather(*[bounded(i) for i in items]))

> **Scale note.** Every child task the orchestrator awaits costs it some memory for tracking.
> For fan-outs beyond ~10k items, batch first (see the micro-batching pattern in
> [04](./04-reusable-containers.ipynb)) and give the orchestrator more memory than the workers.

## 3. Retries and timeouts

- `retries=3` → up to 4 attempts total. Applies to *task* failures (exceptions, OOM, node loss).
- `timeout` bounds two different things, and it pays to bound both:
  - `max_runtime` — how long an attempt may execute
  - `max_queued_time` — how long it may **wait for capacity** before Flyte gives up.
    This is your automatic detector for "the cluster has no free GPUs / the node pool
    can't scale" — instead of a run silently pending for hours, you get a clear failure.

> 💬 **Discuss:** what should `max_queued_time` be for your GPU jobs? Which events in your cluster (spot exhaustion, quota, autoscaler limits) would trip it — and who should hear about it when it fires?

In [ ]:
from datetime import timedelta

from flyte import Timeout


@env.task(
    retries=3,
    timeout=Timeout(
        max_runtime=timedelta(minutes=10),
        max_queued_time=timedelta(minutes=5),
    ),
)
async def flaky_extract(url: str) -> str:
    import random
    if random.random() < 0.3:
        raise ConnectionError(f"transient failure fetching {url}")
    return f"contents of {url}"

## 4. Recovering from OOM with more memory

The classic failure in data pipelines: input sizes vary 100×, so either you over-provision
everything or some runs OOM. In v2 the *task itself* can catch `flyte.errors.OOMError` and
re-invoke with bigger resources — pay for the big pod only when needed:

In [ ]:
import flyte.errors


@env.task
async def transform(rows: int) -> int:
    # allocate ~rows * 1kB to simulate data-dependent memory use
    data = [b"x" * 1024 for _ in range(rows)]
    return len(data)


@env.task
async def resilient_transform(rows: int) -> int:
    try:
        return await transform(rows=rows)
    except flyte.errors.OOMError:
        print("OOM at 1Gi — retrying attempt with 4Gi")
        return await transform.override(
            resources=flyte.Resources(cpu="1", memory="4Gi")
        )(rows=rows)

This composes with `retries`: transient infrastructure failures use the retry budget, while
the OOM path *changes the resources* rather than retrying the same doomed configuration.
In operational reviews this is the cleanest way to separate **platform errors** (node lost,
image pull) from **user-code errors** (OOM at any size, exceptions) — more in
[appendix B](./appendix/B-observability-and-debugging.md).

## 5. Checkpointing inside a task: `@flyte.trace`

Tasks are the unit of *retry*, but long tasks often have expensive internal phases
(API submissions, LLM calls). `@flyte.trace` on an `async` helper makes each call a
**checkpoint**: on retry, completed calls are replayed from the record instead of
re-executed — and each call shows up in the UI timeline.

In [ ]:
@flyte.trace
async def submit_job(batch_id: int) -> str:
    # e.g. POST to an external service; returns a job handle
    await asyncio.sleep(0.1)
    return f"job-{batch_id}"


@flyte.trace
async def wait_for_job(job_id: str) -> int:
    # e.g. poll until the external service finishes
    await asyncio.sleep(0.2)
    return len(job_id)


@env.task(retries=3)
async def submit_and_wait(batch_ids: List[int]) -> List[int]:
    jobs = await asyncio.gather(*[submit_job(b) for b in batch_ids])      # checkpoint(s)
    results = await asyncio.gather(*[wait_for_job(j) for j in jobs])      # checkpoint(s)
    return list(results)

If the pod dies while polling, the retry **skips re-submission** — the `submit_job` results
are replayed from the trace. Rules of thumb:

- Only `async def` helpers can be traced
- Trace non-deterministic/external calls; don't trace cheap pure functions
- Keep traced inputs/outputs small and serializable

## 6. Signaling: pause for external input

`flyte.new_condition` creates a first-class **condition action**: the task pauses until a
human (or system) signals it — no polling loops. Typical use: approve a write to production,
supply a runtime parameter, gate a deployment.

In [ ]:
from datetime import timedelta

import flyte.errors


@env.task
async def gated_publish(report: str) -> str:
    approval = await flyte.new_condition.aio(
        "publish_approval",
        prompt=f"Publish this report? {report[:80]}...",
        data_type=bool,
        timeout=timedelta(hours=4),
    )
    try:
        approved = await approval.wait.aio()
    except flyte.errors.ConditionTimedoutError:
        return "expired: nobody approved within 4h"

    return "published" if approved else "rejected"

While the run is paused you can signal from the UI, the CLI, or Python:

```bash
flyte get condition <run-name>                      # list pending conditions
flyte signal condition <run-name> publish_approval true
```

```python
import flyte.remote
cond = flyte.remote.Condition.get("publish_approval", run_name="...", action_name="gated_publish")
cond.signal(True)
```

`data_type` can be `bool`, `int`, `float`, or `str` — so a condition can also *collect* a
typed value (e.g. a threshold or a deploy reason) and feed it back into the workflow.

## Further reading

- Next: [03-caching-performance-reproducibility](./03-caching-performance-reproducibility.ipynb)
- The micro-batching pattern at scale: [04-reusable-containers](./04-reusable-containers.ipynb)